# Дом и квартира: Qwen2.5-3B-Instruct + QLoRA
## Рубежный контроль 1

**Это воспроизводимый сценарий запуска, а не уже обученная модель.** В поставке нет реальных ответов Qwen и весов. Локально проверены данные и CPU-часть кода. Полный стек T4 нужно проверить ячейками ниже.

В Colab выберите **GPU T4**. Данные — 120 train / 36 validation / 36 test; все синтетические и еще не проверены профильным экспертом. Перед запуском просмотрите `data/DATA_CARD.md` и уникальные эталоны. Не используйте проект как аварийную консультационную службу.

Откройте этот ноутбук в Colab и загрузите полный ZIP проекта в следующей ячейке. Ядро ноутбука запускает скрипты в отдельном Python 3.12.9: конфликтующие системные ML-пакеты Colab не используются. Все ячейки запускать последовательно, не параллельно.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil
from google.colab import files

PROJECT = Path('/content/home_safe_sft')
if not (PROJECT / 'README.md').exists():
    uploaded = files.upload()  # Выберите home_safe_sft_project.zip
    candidates = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Загрузите один ZIP с полным проектом.')
    target = Path('/content').resolve()
    with zipfile.ZipFile(candidates[0]) as z:
        for info in z.infolist():
            resolved = (target / info.filename).resolve()
            if not resolved.is_relative_to(target):
                raise ValueError('Небезопасный путь внутри ZIP.')
        z.extractall(target)
if not (PROJECT / 'src' / 'pipeline.py').exists():
    raise FileNotFoundError('Ожидается /content/home_safe_sft/src/pipeline.py')
os.chdir(PROJECT)
print('Проект:', PROJECT)
print(subprocess.check_output(['nvidia-smi'], text=True))

## 1. Изолированное окружение

Окружение вне папки проекта не попадет в финальный архив. Требуется интернет для Python, пакетов и модели. Это закрепленная конфигурация, а не установка последних версий. Полный список реально установленных пакетов сохранится командой `prepare`.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.6.14'], check=True)
VENV = Path('/content/home_sft_venv')
PY = str(VENV / 'bin' / 'python')
if not Path(PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.12.9', '--seed', str(VENV)], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-r', 'requirements-torch.txt'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([PY, '-m', 'pip', 'check'], check=True)
subprocess.run([PY, '-c', 'import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))'], check=True)

In [ ]:
def run(args, log_name):
    """Run one process; stream and save output; fail loudly on errors."""
    log = PROJECT / 'artifacts' / 'logs' / log_name
    log.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    with log.open('w', encoding='utf-8') as f:
        process = subprocess.Popen([PY, *args], cwd=PROJECT, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            f.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f'Команда завершилась с кодом {code}; журнал: {log}')
    return log

run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], 'colab_unit_tests.log')
run(['-m', 'src.local_checks'], 'colab_local_checks.log')

## 2. Реальная проверка токенизатора и фиксация эксперимента

`prepare` проверяет CUDA, получает commit базы, загружает токенизатор и JSONL через `datasets.load_dataset`, проверяет длины и границы маски loss. При ошибке остановитесь и сохраните ее в журнале; не обходите проверку обрезкой ответов. После `prepare` не меняйте конфигурацию и данные внутри этого эксперимента.

In [ ]:
run(['-m', 'src.pipeline', 'prepare'], 'prepare.log')

## 3. Два пробных ответа базовой модели

Эти два вопроса взяты из validation. В файл попадут **реальные** ответы, параметры и контрольные суммы. Не заменяйте их подготовленными эталонами.

In [ ]:
run(['-m', 'src.pipeline', 'generate', '--split', 'demo_prompts', '--out', 'artifacts/demo_baseline.jsonl'], 'demo_baseline.log')
for line in (PROJECT / 'artifacts/demo_baseline.jsonl').read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    print('\nВОПРОС:', row['question'], '\nОТВЕТ:', row['answer'])

## 4. Бейзлайн на фиксированном test

Сохраните результаты, но не подбирайте гиперпараметры по тестовым ошибкам. Вход модели — только system + user, без эталона, класса риска и чек-листа.

In [ ]:
run(['-m', 'src.pipeline', 'generate', '--split', 'test'], 'baseline_test.log')

## 5. Sanity check QLoRA: два шага

Данный адаптер служит лишь проверкой GPU-стека и сохраняется отдельно. Итоговая генерация откажется принимать smoke-адаптер. При успешном smoke test полный запуск начнется заново с исходной базы.

In [ ]:
run(['-m', 'src.pipeline', 'train', '--smoke'], 'train_smoke.log')

## 6. Полный SFT

3 эпохи, LoRA rank 8, FP16, NF4, gradient checkpointing, micro-batch 1, accumulation 8. Чекпойнт выбирается только по validation loss. Логируется фактическая память, время, число обучаемых параметров и история loss. Не прерывайте специально после smoke test: он не выполняет требование полного дообучения.

In [ ]:
run(['-m', 'src.pipeline', 'train'], 'train_full.log')
assert (PROJECT / 'artifacts/adapter/adapter_model.safetensors').exists()
print((PROJECT / 'artifacts/adapter/experiment.json').read_text(encoding='utf-8'))

### Сохранение весов сразу после обучения

Скачайте резервный архив, чтобы не потерять завершенное обучение при отключении сессии Colab. Он включает адаптер, lock, конфигурацию и журналы, но не промежуточные большие checkpoints.

In [ ]:
backup = Path('/content/home_safe_adapter_backup.zip')
with zipfile.ZipFile(backup, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for sub in ['artifacts/adapter', 'artifacts/logs', 'configs']:
        for p in (PROJECT / sub).rglob('*'):
            if p.is_file():
                z.write(p, str(p.relative_to(PROJECT)))
    z.write(PROJECT / 'artifacts/experiment_lock.json', 'artifacts/experiment_lock.json')
files.download(str(backup))

## 7. Ответы после SFT и сравнение

Генерация использует те же настройки. Оценка требует полных пар и не принимает файлы разных экспериментов. Полученные графики относятся к реальным запускам; графики в `local_checks` относятся только к данным и сконструированным тестам проверяющего правила.

In [ ]:
run(['-m', 'src.pipeline', 'generate', '--split', 'test', '--adapter', 'artifacts/adapter'], 'tuned_test.log')
run(['-m', 'src.evaluate'], 'evaluation.log')

In [ ]:
from IPython.display import display, Markdown, Image
report = (PROJECT / 'reports/results_generated.md').read_text(encoding='utf-8')
display(Markdown(report.split('## Примеры:')[0]))
display(Image(filename=str(PROJECT / 'artifacts/plots/high_risk_checklist_pass.png')))

## 8. Независимый судья

Экспорт содержит 72 отдельных задания в перемешанном порядке. Не показывайте судье `mapping_private.jsonl`. Выберите **другую модель**, фиксируйте ее фактическое название и версию/дату. Рубрика оценивает безопасность, корректность, полезность и критические ошибки. Из-за маленького набора и неидеальности судьи выводы ограничены.

Доступны два пути: API (может быть платным) или веб-интерфейс с ручным сохранением JSON. Ни один путь не подставляет выдуманные оценки.

In [ ]:
run(['-m', 'src.judge', 'export'], 'judge_export.log')
print((PROJECT / 'artifacts/judge/manual_prompts/J0001.txt').read_text(encoding='utf-8'))

### Вариант A — API, по умолчанию выключен

API должен поддерживать Chat Completions, `temperature=0` и JSON mode. Введите полный URL метода, доступную модель и ключ. Проверьте тариф до включения. Ключ не сохраняется в коде или файлах проекта. Первые два задания — проверка формата; затем обрабатываются оставшиеся. При несовместимых параметрах используйте ручной вариант либо адаптируйте и задокументируйте клиент.

In [ ]:
RUN_JUDGE_API = False  # Измените на True только для выбранного API.
if RUN_JUDGE_API:
    from getpass import getpass
    os.environ['JUDGE_API_URL'] = input('Полный HTTPS URL chat completions: ').strip()
    os.environ['JUDGE_MODEL'] = input('Идентификатор модели-судьи: ').strip()
    os.environ['JUDGE_API_KEY'] = getpass('API key: ')
    try:
        run(['-m', 'src.judge', 'run', '--limit', '2'], 'judge_sanity.log')
        run(['-m', 'src.judge', 'run'], 'judge_full.log')
    finally:
        os.environ.pop('JUDGE_API_KEY', None)
else:
    print('API не запускался. Для ручной оценки используйте manual_prompts/*.txt.')

### Вариант B — веб-интерфейс

Передавайте каждый `manual_prompts/J....txt` в отдельный разговор с одной выбранной моделью. Сохраняйте ответ-JSON без изменения баллов, один объект на строку. Полученный `results.jsonl` можно загрузить следующей ячейкой. При неполном наборе отчет покажет покрытие и посчитает сравнение только по полным парам. Не выбирайте только удачные ответы.

In [ ]:
IMPORT_MANUAL_JUDGE = False
if IMPORT_MANUAL_JUDGE:
    manual = files.upload()  # Один JSONL с полученными оценками.
    names = [n for n in manual if n.endswith('.jsonl')]
    if len(names) != 1:
        raise ValueError('Ожидается один JSONL с результатами судьи.')
    dest = PROJECT / 'artifacts/judge/results.jsonl'
    if dest.exists():
        raise FileExistsError('Результаты уже существуют; не смешивайте независимые запуски.')
    dest.write_bytes(manual[names[0]])
    from datetime import datetime, timezone
    metadata = {'transport': 'manual_web_interface',
                'judge_model': input('Название/версия модели-судьи: ').strip(),
                'interface': input('Название интерфейса: ').strip(),
                'date_utc': datetime.now(timezone.utc).isoformat(),
                'parameters': 'Параметры интерфейса не контролировались; указать доступные в отчете.'}
    (PROJECT / 'artifacts/judge/run_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
if (PROJECT / 'artifacts/judge/results.jsonl').exists():
    run(['-m', 'src.judge', 'summarize'], 'judge_summary.log')
    display(Markdown((PROJECT / 'reports/judge_results_generated.md').read_text(encoding='utf-8').split('## Объяснения')[0]))
else:
    print('Независимая оценка не проведена. Не отмечайте этот пункт как выполненный.')

## 9. Отчет и сдача

Обновите `reports/report.md` фактическими таблицами из `results_generated.md` и `judge_results_generated.md`, парами ответов, журналом ошибок и выводами. **PDF в исходном архиве — подготовительная версия; результаты в нем не обновляются автоматически.** Сохраните финальный отчет в объеме 5–10 страниц. Код `src/render_report.py` позволяет повторно собрать PDF из Markdown после установки `requirements-report.txt`: используйте `python -m src.render_report --final --output reports/final.pdf` только после обновления текста и реальных запусков. При запуске из ноутбука вызывайте Python из переменной `PY`.

Разместите весь каталог `artifacts/adapter` в доступном хранилище; заполните `docs/WEIGHTS.md` действительной ссылкой и checksum. Опубликуйте репозиторий с кодом, данными, метриками, журналами и инструкцией, затем передайте ссылку и отчет в LMS. Не публикуйте ключи API. Промежуточные checkpoints не нужны для обычной сдачи.

Ниже сохраняется итоговый архив с полученными артефактами, без большого каталога checkpoints и без smoke-адаптера.

In [ ]:
final_zip = Path('/content/home_safe_sft_results.zip')
with zipfile.ZipFile(final_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in PROJECT.rglob('*'):
        if not p.is_file():
            continue
        rel = p.relative_to(PROJECT)
        if any(part in {'runs', '__pycache__', '.venv', '.ipynb_checkpoints', 'smoke_adapter'} for part in rel.parts):
            continue
        if p.name in {'.env'} or p.suffix in {'.key', '.pyc'}:
            continue
        z.write(p, str(Path('home_safe_sft') / rel))
print('Итоговый архив:', final_zip)
files.download(str(final_zip))